# 당뇨병 검사 결과 분류

## 과제 안내

* 머신러닝 1~7차시에서 배운 내용을 활용하여 당뇨병 검사 결과를 예측하는 **KNN 분류 프로그램**을 만드는 과제입니다. 총 10문제.

* 여러분은 의료 데이터를 분석하는 데이터 분석가입니다.

* 혈당, 혈압, BMI, 인슐린, 나이 등의 정보를 이용하여 당뇨병 검사 결과가 양성인지 음성인지 예측하는 프로그램을 개발해야 합니다.

* 데이터에는 누락된 측정값이 존재하며, 특성마다 숫자의 범위도 다릅니다.

* 판다스로 데이터를 탐색하고 결측치를 처리한 뒤 KNN을 학습하세요. 이후 k값과 스케일링 방법을 변경하여 예측 성능이 어떻게 달라지는지 분석하세요.

**이번 과제에서 사용하는 머신러닝 알고리즘은 KNN 하나입니다.**

- **범위:** 판다스 데이터 처리 + 머신러닝 1~7차시
- **데이터:** Pima Indians Diabetes (768행, 입력 특성 8개, 정답 2개)
- **환경:** VS Code + Jupyter Notebook, `.venv` 커널
- **파일명:** `report1120.ipynb`
- **제출:** GitHub 저장소 링크
- **규칙:** 문제마다 마크다운 셀(`## 문제번호. 제목`)과 코드 셀을 작성하세요.
- **분석 질문:** 각 문제의 마지막에 마크다운 셀로 답변하세요.
- **평가 지표:** 정확도, 정밀도, 재현율, F1, 혼동행렬
- **출력:** 정확도·정밀도·재현율·F1은 소수 넷째 자리까지 출력하세요.

### 과제 목표

다음 전체 과정을 스스로 완성하는 것이 목표입니다.

**데이터 탐색 → 결측치 확인 → 학습·테스트 데이터 분리 → 결측치 처리 → KNN 학습 → 성능 평가 → 과대적합·과소적합 분석 → 스케일링 → 성능 비교 → 최종 결과 분석**

최고 정확도를 기록하는 것보다 각 단계가 왜 필요한지 이해하고 결과를 설명하는 것이 중요합니다.

## 준비 셀

노트북 맨 위에 아래 2개 셀을 만들고 실행하세요.

In [178]:
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [179]:
# 사이킷런을 이용해 당뇨병 데이터 다운로드
diabetes = fetch_openml(
    data_id=37,
    as_frame=True,
    parser="pandas"
)

# 특성 데이터 복사
df = diabetes.data.copy()

# 정답 추가 (0 = 음성, 1 = 양성)
df["target"] = diabetes.target.map({
    "tested_negative": 0,
    "tested_positive": 1
}).astype(int)

print(df.shape)
print(df.head(3))

(768, 9)
   preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1


## 1. 데이터 탐색

In [180]:
# 1. 전체 행 개수와 열 개수를 출력하세요.
print("shape :", df.shape)

# 2. 상위 5행을 출력하세요.
print("head :\n", df.head(5))

# 3. 각 열의 데이터 타입을 확인하세요.
print("dtype :\n", df.dtypes)

# 4. `describe()`로 통계 정보를 확인하세요.
print("describe :\n", df.describe())

# 5. `target`의 값별 데이터 개수를 출력하세요.
print("target_value :", df["target"].value_counts())

minus = (df["target"] == 0).sum()
plus = (df["target"] == 1).sum()
total = len(df)
# 6. 양성(1)과 음성(0)의 비율을 백분율로 출력하세요.
print(f"양성 비율 :", round(plus / total * 100, 4), "%")
print(f"음성 비율 :", round(minus / total * 100, 4), "%")


shape : (768, 9)
head :
    preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1
3     1    89    66    23    94  28.1  0.167   21       0
4     0   137    40    35   168  43.1  2.288   33       1
dtype :
 preg        int64
plas        int64
pres        int64
skin        int64
insu        int64
mass      float64
pedi      float64
age         int64
target      int64
dtype: object
describe :
              preg        plas        pres        skin        insu        mass  \
count  768.000000  768.000000  768.000000  768.000000  768.000000  768.000000   
mean     3.845052  120.894531   69.105469   20.536458   79.799479   31.992578   
std      3.369578   31.972618   19.355807   15.952218  115.244002    7.884160   
min      0.000000    0.000000    0.000000    0.000000    0.000000    0.000000   
25%      1.000000   99.00

## 분석 질문
1. 전체 데이터에서 양성(1)의 비율은 몇 %인가요?
- 34.8958%입니다.
2. 입력 특성 중 숫자 범위가 가장 큰 특성과 가장 작은 특성은 무엇인가요?
- 가장 큰 특성은 insu(인슐린)이고 0 ~ 846까지 분포합니다.
- 가장 작은 특성은 pedi(당뇨병 가족력 관련 지표)이고 0.078 ~ 2.42까지 분포합니다.
3. 특성마다 숫자 범위가 다른 것이 KNN에 어떤 영향을 줄 수 있을까요?
- 숫자 범위에 따라 숫자가 큰 특성은 작은 변화로도 값이 크게 벌어지고, 
- 숫자가 작은 특성은 큰 변화가 있어도 거리 차이가 거의 변화하지 않습니다. 

# 2. 결측치 탐색

In [181]:
# 데이터에서 누락된 측정값을 찾아보세요.

# 1. `isna().sum()`으로 결측치 개수를 출력하세요.
print("first_isna :\n", df.isna().sum(), "\n")

# 2. 준비 셀에서 안내한 5개 열의 0 개수를 각각 출력하세요.
five_cols = ["plas", "pres", "skin", "insu", "mass"]
for col in five_cols:
    print(f"{col}: {(df[col] == 0).sum()}")

# 3. 해당 5개 열에서 0을 `NaN`으로 변경하세요.
df[five_cols] = df[five_cols].replace(0, None)

# 4. 다시 `isna().sum()`으로 결측치 개수를 확인하세요.
print("second_isna :\n", df.isna().sum())

# 5. 결측치가 하나라도 포함된 행의 개수를 출력하세요.
print("line_isna :",(df.isna().sum(axis=1)>0).sum())

# **주의:** 이 단계에서는 결측치를 삭제하거나 다른 값으로 채우지 않습니다.

first_isna :
 preg      0
plas      0
pres      0
skin      0
insu      0
mass      0
pedi      0
age       0
target    0
dtype: int64 

plas: 5
pres: 35
skin: 227
insu: 374
mass: 11
second_isna :
 preg        0
plas        5
pres       35
skin      227
insu      374
mass       11
pedi        0
age         0
target      0
dtype: int64
line_isna : 376


## 분석 질문
1. 처음 `isna()`를 실행했을 때 결측치가 발견되지 않은 이유는 무엇인가요?
- 데이터가 NaN 이나 None 이 아니고 0으로 입력되어 있어 결측치가 발견되지 않았습니다.
2. 모든 열의 0을 `NaN`으로 변경하면 안 되는 이유는 무엇인가요?
- preg(임신횟수)와 같은 데이터는 0 일수 있는 실제 데이터 이기에 해당 데이터 까지 변경하면 안됩니다.

# 3. 특성과 정답 분리 및 데이터 분할

머신러닝을 학습하기 위한 데이터를 준비하세요.

1. `target`을 제외한 열들을 `X`에 저장하세요.
2. `target`을 `y`에 저장하세요.
3. `X`와 `y`의 shape를 출력하세요.
4. `train_test_split()`으로 데이터를 분리하세요.

**분할 조건**

- 학습 데이터 80%
- 테스트 데이터 20%
- `random_state=42`
- `stratify=y`
1. `X_train`, `X_test`, `y_train`, `y_test`의 shape를 출력하세요.
2. 학습 데이터와 테스트 데이터의 양성·음성 개수를 각각 출력하세요.

In [182]:
X = df.drop(columns="target")
y = df["target"]

print("X :", X.shape)
print("y :", y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test :", y_test.shape)

print("학습 데이터 - 음성 :", (y_train == 0).sum(), "양성 :", (y_train == 1).sum())
print("학습 데이터 - 음성 :", (y_test == 0).sum(), "양성 :", (y_test == 1).sum())

X : (768, 8)
y : (768,)
X_train : (614, 8)
X_test : (154, 8)
y_train : (614,)
y_test : (154,)
학습 데이터 - 음성 : 400 양성 : 214
학습 데이터 - 음성 : 100 양성 : 54


## 분석 질문

1. 학습 데이터와 테스트 데이터를 나누는 이유는 무엇인가요?
- 모델이 학습 데이터를 외우는게 아니라 새로운 데이터도 얼마나 예측을 잘하는지 평가하기 위해서입니다.
2. `stratify=y`를 사용하는 이유는 무엇인가요?
- 분할을 했을때에 양성이나 음성 값이 극단적으로 크거나 적게 들어가는 상황을 방지해 예측 정확도를 높입니다. 
3. `random_state`를 지정하면 어떤 장점이 있나요?
- 결과 값을 난수 시드를 맞춰 실행해 항상 같은 값을 볼수 있다는 장점이 있습니다. 

# 4. 결측치 처리

3번에서 분리한 학습 데이터와 테스트 데이터의 결측치를 처리하세요.

**조건**

- 결측치는 각 특성의 중앙값(`median`)으로 채우세요.
- 중앙값은 학습 데이터에서만 계산하세요.
- 테스트 데이터에는 학습 데이터에서 계산한 중앙값을 사용하세요.
- 원본 데이터는 유지하고 결측치를 처리한 데이터를 별도의 변수에 저장하세요.

다음 작업을 수행하세요.

1. 학습 데이터에서 각 특성의 중앙값을 계산하세요.
2. 학습 데이터의 결측치를 중앙값으로 채우세요.
3. 테스트 데이터의 결측치도 동일한 중앙값으로 채우세요.
4. 처리 후 학습 데이터와 테스트 데이터의 결측치 개수를 `isna().sum()`으로 출력하세요.
5. 처리 전후 데이터의 행 개수를 비교하세요.

In [183]:
train_medians = X_train.median()
# print("train_medians :\n", train_medians)

X_train_med = X_train.fillna(train_medians)
X_test_med = X_test.fillna(train_medians)

print("isna_X_train\n", X_train_med.isna().sum())
print("isna_X_test", X_test_med.isna().sum())

print("학습 데이터 - 처리 전 :", len(X_train), "처리 후 :", len(X_train_med))
print("테스트 데이터 - 처리 전 :", len(X_test), "처리 후 :",len(X_test_med))

isna_X_train
 preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64
isna_X_test preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64
학습 데이터 - 처리 전 : 614 처리 후 : 614
테스트 데이터 - 처리 전 : 154 처리 후 : 154


## 분석 질문

1. 결측치를 삭제하지 않고 중앙값으로 채우면 어떤 장점이 있나요?
- 결측치가 많은 값일 수록 모두 삭제시 샘플에 남은 값이 적어져 한쪽에 치우친 결과가 출력될 수 있습니다. 
- 그래서 중앙값을 채워 다른 특성과의 활용도를 높일 수 있습니다. 
2. 테스트 데이터의 중앙값을 따로 계산해서 사용하면 안 되는 이유는 무엇인가요?
- 테스트 데이터는 평가를 위한 데이터여서 해당값의 중앙값은 사실상 모르는 값의 중앙값을 구하는 것과 같습니다. 
- 지금은 시드 고정했지만 매번 랜덤 값이 들어오면 중앙값의 기준점이 달라져 매번 결과가 크게 차이가 있을겁니다.

# 5. 첫 번째 KNN 학습

결측치 처리가 완료된 데이터를 이용하여 KNN을 학습하세요.

**조건**

- `KNeighborsClassifier` 사용
- `n_neighbors=5`
- 스케일링하지 않은 데이터 사용
- 4번에서 결측치를 처리한 학습·테스트 데이터 사용

다음 작업을 수행하세요.

1. KNN 객체를 생성하세요.
2. `fit()`으로 학습하세요.
3. 학습 정확도를 출력하세요.
4. 테스트 정확도를 출력하세요.
5. `predict()`로 테스트 데이터의 결과를 예측하세요.
6. 예측 결과와 실제 정답을 각각 앞 10개씩 출력하세요.

In [184]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_med, y_train)
train_score = round(model.score(X_train_med, y_train), 4)
test_score = round(model.score(X_test_med, y_test), 4)

print("train_score :", train_score)
print("test_score :", test_score)

pred = model.predict(X_test_med)
print("예측 결과 :", pred[:10])
print("실제 정답 :", y_test[:10].values)


train_score : 0.8013
test_score : 0.6753
예측 결과 : [1 0 0 0 0 0 0 1 0 1]
실제 정답 : [0 0 0 1 0 0 1 1 0 0]


## 분석 질문

1. KNN에서 k=5는 어떤 의미인가요?
- KNN 알고리즘은 새 데이터와 가장 가까운 k개의 데이터를 비교하는 거라 주변의 5개의 데이터를 찾아 다수결로 클래스를 결정한다는 의미입니다.
2. 학습 정확도와 테스트 정확도는 각각 무엇을 의미하나요?
- 학습 정확도 : 학습에 쓴 데이터가 얼마나 맞췄는지의 정도
- 테스트 정확도 : 학습에 안 쓴 데이터가 얼마나 맞췄는지의 정도
3. 학습 정확도와 테스트 정확도에 차이가 발생하는 이유는 무엇인가요?
- 테스트 정확도는 처음 보는 데이터와 같아 경계나 값이 몰려 있는 부분에서의 오차가 발생해 정확도에 차이가 있습니다.

# 6. KNN 성능 평가

5번에서 학습한 KNN의 예측 결과를 평가하세요.

다음 항목을 출력하세요.

1. 혼동행렬 (`confusion_matrix`)
2. 정확도 (`accuracy_score`)
3. 정밀도 (`precision_score`)
4. 재현율 (`recall_score`)
5. F1 점수 (`f1_score`)
6. `classification_report`

이번 데이터는 이진 분류이므로 `0`은 음성, `1`은 양성입니다. 정밀도와 재현율은 양성(1)을 기준으로 계산하세요.

In [185]:
print("1. 혼동행렬 :\n", confusion_matrix(y_test, pred))
print("2. 정확도 :", round(accuracy_score(y_test, pred), 4))
print("3. 정밀도 :", precision_score(y_test, pred))
print("4. 재현율 :", recall_score(y_test, pred))
print("5. F1 점수 :", round(f1_score(y_test, pred), 4))
print("6. classification_report :\n", classification_report(y_test, pred))

1. 혼동행렬 :
 [[77 23]
 [27 27]]
2. 정확도 : 0.6753
3. 정밀도 : 0.54
4. 재현율 : 0.5
5. F1 점수 : 0.5192
6. classification_report :
               precision    recall  f1-score   support

           0       0.74      0.77      0.75       100
           1       0.54      0.50      0.52        54

    accuracy                           0.68       154
   macro avg       0.64      0.64      0.64       154
weighted avg       0.67      0.68      0.67       154




## 분석 질문

1. 실제 양성인데 음성으로 잘못 예측한 데이터(FN)는 몇 개입니까?
- 27개 입니다. 
2. 실제 음성인데 양성으로 잘못 예측한 데이터(FP)는 몇 개입니까?
- 23개 입니다. 
3. 정확도는 높지만 재현율은 낮은 상황이 발생할 수 있나요?
- 정확도( 77+27 / 154) 가 재현율 (27 / 54) 가 낮을 수 있습니다. 
- 음성을 많이 맞추면 실제 양성을 놓칠수 있는 확률이 높아지기 때문입니다. 
4. 당뇨병 양성을 놓치는 것을 줄이려면 어떤 평가 지표를 중요하게 봐야 할까요?
- 양성을 놓치는 것이라면 양성인데 음성으로 판정하는 경우이기 때문에 FN 값을 줄여야 합니다. 
- 그래서 재현율을 중요히 봐야 합니다. 

# 7. k값에 따른 성능 비교

KNN의 k값을 변경하면서 학습 정확도와 테스트 정확도가 어떻게 변하는지 확인하세요.

실험할 k값:

`1, 3, 5, 7, 10, 15, 20, 30`

**조건**

- 반복문을 사용하세요.
- 스케일링하지 않은 데이터를 사용하세요.
- 4번에서 결측치를 처리한 데이터를 사용하세요.
- k값 외에는 동일한 조건을 유지하세요.

각 k값의 학습 정확도와 테스트 정확도를 출력하고 다음 표를 완성하세요.

마크다운 또는 코드셀에 주석으로 적어주시면 됩니다.
e.g. 
1 / 0.91 / 0.93 
3 / 0.92 / 0.94
이런식으로 ,,,,


In [186]:
k_list = [1, 3, 5, 7, 10, 15, 20, 30]

print("k  | 학습 정확도 | 테스트 정확도")

for k in k_list:
    model_k = KNeighborsClassifier(n_neighbors=k)
    model_k.fit(X_train_med, y_train)
    
    model_train = round(model_k.score(X_train_med, y_train), 4)
    model_test = round(model_k.score(X_test_med, y_test), 4)
    
    print(k ,"|", model_train, "\t|", model_test)

k  | 학습 정확도 | 테스트 정확도
1 | 1.0 	| 0.6818
3 | 0.842 	| 0.6818
5 | 0.8013 	| 0.6753
7 | 0.785 	| 0.6753
10 | 0.798 	| 0.6688
15 | 0.7915 	| 0.6558
20 | 0.7883 	| 0.7208
30 | 0.7655 	| 0.7208


* k	학습 정확도	테스트 정확도
* 1   /   1       /   0.6818		
* 3	/	0.842   /   0.6818
* 5	/   0.8013	/   0.6753
* 7	/   0.785   /   0.6753	
* 10	/   0.798   /   0.6688 	
* 15	/   0.7915  /   0.6558	
* 20	/   0.7883  /   0.7208	
* 30	/   0.7655  /   0.7208	

## 분석 질문

1. 학습 정확도가 가장 높은 k값은 무엇인가요?
- 학습 정확도라 본인과 가장 가까운 k = 1이 가장 높습니다. 
2. 테스트 정확도가 가장 높은 k값은 무엇인가요?
- k = 20, 30 두 값이 0.7208로 가장 높습니다. 
3. k가 작을 때 과대적합이 발생하기 쉬운 이유는 무엇인가요?
- k가 작을수록 가까운 데이터에만 의존해 학습 정확도는 높지만 테스트 정확도에서는 성능이 낮은 과대덕합이 발생하기 쉽습니다.  
4. k가 지나치게 커지면 과소적합이 발생할 수 있는 이유는 무엇인가요?
- k가 지나치게 커지면 너무 여러개의 대이터와 영향을 받게 되어 경계가 단순해집니다. 그래서 학습 정확도와 테스트 정확도 모두 낮은 과소적합이 발생하기 쉽습니다. 
5. 학습 정확도만 보고 k값을 선택해도 될까요?
- 아니요. 학습 정확도가 아닌 테스트 정확도를 보고 판별해야 합니다. 학습 정확도는 본인이 얼마나 잘외웠는를 판별합니다. 

# 8. StandardScaler 적용



이번에는 `StandardScaler`로 특성을 표준화한 뒤 동일한 KNN을 학습하세요.

**조건**

- 4번에서 결측치를 처리한 데이터 사용
- `StandardScaler` 사용
- 스케일러는 학습 데이터에만 `fit()`
- 학습 데이터와 테스트 데이터는 같은 스케일러로 변환
- KNN의 k값은 5로 고정

다음 작업을 수행하세요.

1. `StandardScaler` 객체를 생성하세요.
2. 학습 데이터로 `fit()` 하세요.
3. 학습 데이터와 테스트 데이터를 각각 `transform()` 하세요.
4. 변환된 학습 데이터를 DataFrame으로 만들고 `describe()`를 출력하세요.
5. 변환된 데이터로 KNN(k=5)을 학습하세요.
6. 학습 정확도와 테스트 정확도를 출력하세요.
7. 재현율을 출력하세요.

In [187]:
scaler = StandardScaler()
scaler.fit(X_train_med)

X_train_scaler = scaler.transform(X_train_med)
X_test_scaler = scaler.transform(X_test_med)

X_train_scaled_df = pd.DataFrame(X_train_scaler, columns=X.columns)    
print(X_train_scaled_df.describe())

scaler_model = KNeighborsClassifier(n_neighbors=5)
scaler_model.fit(X_train_scaler, y_train)

scaler_model_train_score = round(scaler_model.score(X_train_scaler, y_train), 4)
scaler_model_test_score = round(scaler_model.score(X_test_scaler, y_test), 4)

print("학습 정확도 :", scaler_model_train_score)
print("테스트 정확도 :", scaler_model_test_score)

scaler_pred = scaler_model.predict(X_test_scaler)
recall_pred = round(recall_score(y_test, scaler_pred), 4)
print("재현율 :", recall_pred)

               preg          plas          pres          skin          insu  \
count  6.140000e+02  6.140000e+02  6.140000e+02  6.140000e+02  6.140000e+02   
mean  -6.943414e-17 -1.099374e-16  3.095606e-16 -3.471707e-17 -4.339634e-18   
std    1.000815e+00  1.000815e+00  1.000815e+00  1.000815e+00  1.000815e+00   
min   -1.153338e+00 -2.190541e+00 -3.924957e+00 -2.480958e+00 -1.559139e+00   
25%   -8.513551e-01 -7.562208e-01 -6.636760e-01 -4.549828e-01 -2.249692e-01   
50%   -2.473897e-01 -1.558076e-01 -1.141979e-02 -4.766137e-03 -1.614373e-01   
75%    6.585583e-01  6.113872e-01  6.408364e-01  3.328964e-01 -9.790537e-02   
max    3.980368e+00  2.579408e+00  4.065181e+00  7.874025e+00  7.703814e+00   

               mass          pedi           age  
count  6.140000e+02  6.140000e+02  6.140000e+02  
mean   1.649061e-16 -1.099374e-16 -1.084908e-16  
std    1.000815e+00  1.000815e+00  1.000815e+00  
min   -2.089621e+00 -1.192094e+00 -1.045895e+00  
25%   -7.073644e-01 -7.042617e-01 -7.9

## 분석 질문

1. 표준화 후 각 특성의 평균과 표준편차는 대략 얼마인가요?
- 각 특성의 평균과 표준편차는 대략 평균은 0 표준편차는 1 근처 값입니다.
- 위의 값들이 너무 작게 보여 찾아보니 미세한 오차가 있다고 합니다.  
2. 5번에서 구한 스케일링 전 정확도와 비교하면 어떤 차이가 있나요?
- 학습 정확도 : 0.8013 / 테스트 정확도 : 0.6753
- 학습 정확도 : 0.829 / 테스트 정확도 : 0.7532
- 학습 정확도와 테스트 정확도 모두 스케일링 이후 상승하였습니다.
3. 모델의 알고리즘과 k값은 그대로인데 정확도가 달라지는 이유는 무엇인가요?
- knn은 데이터간 거리를 계산하는 방식이고 스케일링을 통해 퍙균과 표준편차를 맞추게 되어 거리 계산이 균등해져 처음 k값의 데이터가 아닌 샘플 선정이 이뤄져 정확도가 달라집니다..?

# 9. MinMaxScaler 적용 및 비교

이번에는 `MinMaxScaler`를 적용하고 KNN의 성능을 확인하세요.

**조건**

- 4번에서 결측치를 처리한 데이터 사용
- `MinMaxScaler` 사용
- 학습 데이터에만 `fit()`
- KNN의 k값은 5로 고정

다음 작업을 수행하세요.

1. `MinMaxScaler`로 학습·테스트 데이터를 변환하세요.
2. 변환된 학습 데이터의 `describe()`를 출력하세요.
3. 변환된 데이터로 KNN(k=5)을 학습하세요.
4. 학습 정확도와 테스트 정확도를 출력하세요.
5. 재현율을 출력하세요.
6. 세 가지 전처리 방법의 성능을 비교하세요.

In [188]:
minmax = MinMaxScaler()
minmax.fit(X_train_med)

X_train_minmax = minmax.transform(X_train_med)
X_test_minmax = minmax.transform(X_test_med)

X_train_minmax_df = pd.DataFrame(X_train_minmax, columns=X.columns)    
print(X_train_minmax_df.describe())

minmax_model = KNeighborsClassifier(n_neighbors=5)
minmax_model.fit(X_train_minmax, y_train)

minmax_model_train_score = round(minmax_model.score(X_train_minmax, y_train), 4)
minmax_model_test_score = round(minmax_model.score(X_test_minmax, y_test), 4)

print("minmax 학습 정확도 :", minmax_model_train_score)
print("minmax 테스트 정확도 :", minmax_model_test_score)

minmax_pred = scaler_model.predict(X_test_minmax)
minmax_recall_pred = round(recall_score(y_test, minmax_pred), 4)
print("minmax 재현율 :", minmax_recall_pred)

print("Standard 학습 정확도 :", scaler_model_train_score)
print("Standard 테스트 정확도 :", scaler_model_test_score)
print("Standard 재현율 :", recall_pred)

print("학습 정확도 :", train_score)
print("테스트 정확도 :", test_score)
pred_recall = round(recall_score(y_test, pred), 4)
print("재현율 :", pred_recall)

             preg        plas        pres        skin        insu        mass  \
count  614.000000  614.000000  614.000000  614.000000  614.000000  614.000000   
mean     0.224660    0.459238    0.491225    0.239591    0.168320    0.291374   
std      0.194950    0.209817    0.125256    0.096651    0.108045    0.139553   
min      0.000000    0.000000    0.000000    0.000000    0.000000    0.000000   
25%      0.058824    0.300699    0.408163    0.195652    0.144033    0.192740   
50%      0.176471    0.426573    0.489796    0.239130    0.150892    0.290389   
75%      0.352941    0.587413    0.571429    0.271739    0.157750    0.374233   
max      1.000000    1.000000    1.000000    1.000000    1.000000    1.000000   

             pedi         age  
count  614.000000  614.000000  
mean     0.175246    0.206107  
std      0.147127    0.197224  
min      0.000000    0.000000  
25%      0.071715    0.050000  
50%      0.132962    0.133333  
75%      0.247327    0.333333  
max      1.000

## 분석 질문

1. 테스트 정확도가 가장 높은 전처리 방법은 무엇인가요?
- StandardScaler 방식이 가장 높습니다. 
2. 재현율이 가장 높은 전처리 방법은 무엇인가요?
- MinMaxScaler 방식이 가장 높습니다. 
3. 정확도가 높은 방법과 재현율이 높은 방법이 다를 수도 있나요?
- 정확도는 (맞춘 비율 / 전체) 재현율은 실제 양성 중 양성으로 예측한 비율이기 때문에 다를 수 있습니다. 
4. 스케일링을 적용한다고 해서 정확도가 반드시 높아지는 것은 아닌 이유는 무엇일까요?
- 각 스케일링 방식을 통해 데이터 값의 범위를 맞춰주는 것이라 데이터의 값이 너무 극단적일 때에는 정확도가 더 떨어 질수 있습니다. 

# 10. 종합 — KNN 설정 선택 및 최종 분석

지금까지 실험한 결과를 이용하여 가장 적절하다고 생각하는 KNN 설정을 선택하세요.

## 1단계 — 스케일링 방법 선택

다음 중 하나를 선택하세요.

- 스케일링 없음
- StandardScaler
- MinMaxScaler


In [189]:
last_scaler = MinMaxScaler()
last_scaler.fit(X_train_med)

X_train_last = last_scaler.transform(X_train_med)
X_test_last = last_scaler.transform(X_test_med)

## 2단계 — k값 실험

선택한 스케일링 방법으로 다음 k값을 실험하세요.

`1, 3, 5, 7, 10, 15, 20, 30`

각 k값에 대해 다음 결과를 표로 정리하세요.

- 학습 정확도
- 테스트 정확도
- 재현율

In [190]:
last_list = [1, 3, 5, 7, 10, 15, 20, 30]

print("k  | 학습 정확도 | 테스트 정확도 | 재현율")

for k in last_list:
    last_model = KNeighborsClassifier(n_neighbors=k)
    last_model.fit(X_train_last, y_train)
    
    last_model_train = round(last_model.score(X_train_last, y_train), 4)
    last_model_test = round(last_model.score(X_test_last, y_test), 4)

    last_pred = last_model.predict(X_test_last)
    last_recall_pred = round(recall_score(y_test, last_pred), 4)
    
    print(k ,"|", last_model_train, "\t|", last_model_test, "\t|",last_recall_pred)

k  | 학습 정확도 | 테스트 정확도 | 재현율
1 | 1.0 	| 0.7013 	| 0.5556
3 | 0.8713 	| 0.7403 	| 0.5926


5 | 0.8322 	| 0.7338 	| 0.5556
7 | 0.8306 	| 0.7273 	| 0.5556
10 | 0.7915 	| 0.7338 	| 0.463
15 | 0.7997 	| 0.7273 	| 0.537
20 | 0.7915 	| 0.7338 	| 0.463
30 | 0.7866 	| 0.7403 	| 0.5


## 3단계 — 최종 설정 선택

실험 결과를 비교하여 다음 두 가지를 결정하세요.

1. 사용할 스케일링 방법
2. 사용할 k값

단순히 정확도가 가장 높은 설정을 선택해도 되지만, 재현율이나 과대적합·과소적합 가능성을 고려하여 다른 설정을 선택해도 됩니다.

In [191]:
# 사용할 스케일링 방법 : Minmax, 사용할 k값 : 3
best_model = KNeighborsClassifier(n_neighbors=3)
best_model.fit(X_train_last, y_train)

best_train_score = round(best_model.score(X_train_last, y_train), 4)
best_test_score = round(best_model.score(X_test_last, y_test), 4)
best_pred = best_model.predict(X_test_last)

best_cm = confusion_matrix(y_test, best_pred)
best_prec = round(precision_score(y_test, best_pred), 4)
best_rec = round(recall_score(y_test, best_pred), 4)
best_f1 = round(f1_score(y_test, best_pred), 4)

## 4단계 — 선택한 설정으로 KNN 학습 및 평가

선택한 설정으로 KNN을 학습하고 다음 결과를 출력하세요.

1. 학습 정확도
2. 테스트 정확도
3. 혼동행렬
4. 정밀도
5. 재현율
6. F1 점수

In [192]:
print("학습 정확도 :", best_train_score)
print("테스트 정확도 :", best_test_score)
print("혼동행렬 :\n", best_cm)
print("정밀도 :", best_prec)
print("재현율 :", best_rec)
print("F1 점수 :", best_f1)


학습 정확도 : 0.8713
테스트 정확도 : 0.7403
혼동행렬 :
 [[82 18]
 [22 32]]
정밀도 : 0.64
재현율 : 0.5926
F1 점수 : 0.6154


## 5단계 — 최종 분석 보고서

마크다운 셀에 다음 질문의 답변을 작성하세요.

1. 최종적으로 어떤 스케일링 방법과 k값을 선택했나요?
- MinMaxScaler 스케일링 방식과, k 값은 3을 선택했습니다. 
2. 해당 설정을 선택한 이유는 무엇인가요?
- 학습 정확도가 과대 적합인 1을 제외하고, 재현율이 낮은 10, 20, 30을 제외한 후
- 테스트 정확도와 재현율이 가장 높은 3을 선택했습니다. 
3. 스케일링 전후 KNN의 성능은 어떻게 달라졌나요?
- 스케일링 미적용 기본 값은 학습 정확도 : 0.8013 | 테스트 정확도 : 0.6753 | 재현율 : 0.5입니다. 
- 스케일링 이후 학습 정확도 : 0.8713 | 테스트 정확도 : 0.7403 | 재현율 : 0.5926로 성능이 개선되었습니다.
4. k값이 달라지면서 정확도는 어떤 변화를 보였나요?
- 학습 정확도는 k값이 늘어날수록 정확도가 낮아졌고 테스트 정확도는 의 균등하였으나 k값이 3, 30일때 가장 높았습니다. 
5. 정확도와 재현율 중 어떤 지표를 더 중요하게 고려했나요? 이유를 설명하세요.
- 재현율을 중요하게 고려했습니다. 양성을 음성으로 잘못 예측하는 경우가 줄어야 생존률을 높일 수 있다 생각해서 입니다. 
6. 결측치 처리와 스케일링에서 학습 데이터에만 기준을 맞춰야 하는 이유는 무엇인가요?
- 테스트 데이터는 기존의 학습 데이터에서는 모르는 데이터 값이여야 하기 때문입니다. 테스트 데이터를 기준으로하는 정보를 전처리에 포함하면 테스트 성능이 과하게 표현될 수 있습니다. 
7. 이번 과제를 진행하면서 머신러닝 전체 과정 중 가장 중요하다고 생각한 단계는 무엇인가요?
- 해당 과정에서 여러번의 모델 학습, 예측, 모델 평가과정을 거쳤지만 데이터 준비 과정이 가장 중요하다고 생각합니다. 전처리 과정을 완벽히 마쳐야 이후에 진행되는 과정또한 오류 없이 정확한 값을 표기해 줄 수 있기 때문입니다. 